Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """
WITH L0_disposition AS (
    SELECT
        da.ticket_uuid,
        MAX(da.question_reference_id) AS question_reference_id,
        MAX(o.text) AS text
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON element_at(split(da.option_reference_ids, '"'), 2) = o.reference_id
    WHERE da.question_reference_id IN (
        '3082565b-cc22-47f6-8f3d-fe21a6b2ba6c',
        'babcfbfb-efb5-4528-b22e-55ec5bf0c6ba',
        '465a45b1-4250-49af-8f4e-c8e2564a5d71'
    )
    GROUP BY da.ticket_uuid
),

when_renew AS (
    SELECT
        da.ticket_uuid,
        MAX(da.question_reference_id) AS question_reference_id,
        MAX(o.text) AS text
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON element_at(split(da.option_reference_ids, '"'), 2) = o.reference_id
    WHERE da.question_reference_id IN ('c19b30ff-22f1-423f-9bc2-77b04eb3d29d')
    GROUP BY da.ticket_uuid
),

why_followup AS (
    SELECT
        da.ticket_uuid,
        MAX(da.question_reference_id) AS question_reference_id,
        MAX(o.text) AS text
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON element_at(split(da.option_reference_ids, '"'), 2) = o.reference_id
    WHERE da.question_reference_id IN ('af1e0b77-f9b5-4069-bf87-83271f16ea6a')
    GROUP BY da.ticket_uuid
),

what_done AS (
    SELECT
        da.ticket_uuid,
        MAX(da.question_reference_id) AS question_reference_id,
        MAX(o.text) AS text
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON element_at(split(da.option_reference_ids, '"'), 2) = o.reference_id
    WHERE da.question_reference_id IN ('2323a7b7-62d2-469a-967b-4c4f531c6a6b')
    GROUP BY da.ticket_uuid
),

connected_calls AS (
    SELECT
        cl.call_group_id AS uuid,
        ce.call_id,
        replace(u.name, '.', ' ') AS name,
        CASE WHEN ij.call_id IS NOT NULL THEN 'CONNECTED' ELSE 'Other' END AS call_connection_status,
        MAX(
            CASE
                WHEN get_json_object(ce.attributes, '$.callStatus') = 'CONNECTED'
                THEN to_timestamp(substring(get_json_object(ce.attributes, '$.callConnectedTime'), 1, 19), 'yyyy/MM/dd HH:mm:ss')
                ELSE NULL
            END
        ) AS call_connected_ist,
        MAX(
            CASE
                WHEN event_type = 'CALL_SUMMARY'
                THEN CAST(element_at(split(get_json_object(ce.attributes, '$.talkTime'), ':'), 1) AS INT) * 3600
                   + CAST(element_at(split(get_json_object(ce.attributes, '$.talkTime'), ':'), 2) AS INT) * 60
                   + CAST(element_at(split(get_json_object(ce.attributes, '$.talkTime'), ':'), 3) AS INT)
                ELSE NULL
            END
        ) AS call_duration,
        MAX(CASE WHEN event_type = 'CALL_SUMMARY' THEN get_json_object(ce.attributes, '$.numAttempts') ELSE NULL END) AS AttemptNum,
        MAX(
            CASE
                WHEN event_type = 'CALL_SUMMARY'
                THEN replace(element_at(split(get_json_object(ce.attributes, '$.userId'), '@'), 1), '.', ' ')
                ELSE NULL
            END
        ) AS Agent_name,
        MAX(CASE WHEN event_type = 'CALL_SUMMARY' THEN get_json_object(ce.attributes, '$.recordingUrl') ELSE NULL END) AS recordingUrl
    FROM graham_bell_service.call_events ce
    LEFT JOIN (
        SELECT DISTINCT call_id
        FROM graham_bell_service.call_events
        WHERE event_type = 'DISPOSITION_SUBMITTED'
    ) ij
        ON ij.call_id = ce.call_id
    LEFT JOIN graham_bell_service.call_logs AS cl
        ON ce.call_id = cl.call_id
    LEFT JOIN mjolnir_service.users AS u
        ON cl.navi_agent_id = u.account_id
    WHERE cl.tenant_identifier = 'GI_RENEWAL_CALLING'
    GROUP BY cl.call_group_id, ce.call_id, replace(u.name, '.', ' '), CASE WHEN ij.call_id IS NOT NULL THEN 'CONNECTED' ELSE 'Other' END
),

attempt_time AS (
    SELECT
        ticket_uuid,
        MIN_BY(call_initiation_time, attempt_number) AS first_initiation_time
    FROM (
        SELECT
            cl.call_group_id AS ticket_uuid,
            CASE
                WHEN ce.event_type = 'CALL_SUMMARY'
                THEN CAST(get_json_object(ce.attributes, '$.numAttempts') AS INT)
                ELSE NULL
            END AS attempt_number,
            CASE
                WHEN ce.event_type = 'CALL_SUMMARY'
                THEN to_timestamp(
                    substring(get_json_object(ce.attributes, '$.callInitiationTime'), 1, 19),
                    'yyyy/MM/dd HH:mm:ss'
                )
                ELSE NULL
            END AS call_initiation_time
        FROM graham_bell_service.call_events ce
        LEFT JOIN graham_bell_service.call_logs AS cl
            ON cl.call_id = ce.call_id
        WHERE ce.event_type = 'CALL_SUMMARY'
          AND cl.tenant_identifier = 'GI_RENEWAL_CALLING'
    ) t
    GROUP BY ticket_uuid
),

main_table AS (
    SELECT
        from_utc_timestamp(to_timestamp(t.created_at), 'Asia/Kolkata') AS ticket_creation_ist,
        t.uuid AS crm_ticket_id,
        t.priority,
        COALESCE(cc.call_connection_status, 'Other') AS call_connection_status,
        t.reporter_id AS navi_customer_id,
        to_date(cc.call_connected_ist) AS call_connect_dt,
        COALESCE(cc.name, cc.Agent_name) AS Agent_name,
        cc.call_duration,
        get_json_object(t.meta_data, '$.policyId') AS policy_id,
        get_json_object(t.meta_data, '$.renewalType') AS renewal_type,
        att.first_initiation_time,
        to_date(crd.policy_renewal_ts) AS policy_renewed_dt,
        date_add(to_date(ps.policy_issued_created_at), 365) AS policy_renewal_dt,
        COALESCE(
            to_date(crd.policy_renewal_ts),
            MAX(to_date(crd.policy_renewal_ts)) OVER (PARTITION BY get_json_object(t.meta_data, '$.policyId'))
        ) AS policy_renewed_dt_helper,
        date_sub(date_add(to_date(ps.policy_issued_created_at), 365), 62) AS call_window_starts_from,
        l0d.text AS L0_disposition
    FROM crm_tickets.tickets t
    LEFT JOIN L0_disposition AS l0d
        ON t.uuid = l0d.ticket_uuid
    LEFT JOIN connected_calls AS cc
        ON t.uuid = cc.uuid
    LEFT JOIN why_followup AS fl
        ON t.uuid = fl.ticket_uuid
    LEFT JOIN when_renew AS renew
        ON t.uuid = renew.ticket_uuid
    LEFT JOIN what_done AS wd
        ON t.uuid = wd.ticket_uuid
    LEFT JOIN attempt_time AS att
        ON att.ticket_uuid = t.uuid
    LEFT JOIN corp_analytics.gi_renewal_calling_ticket_datamodel_tbl crd
        ON crd.policy_id = get_json_object(t.meta_data, '$.policyId')
    LEFT JOIN corp_analytics.gi_policy_sales_fct_view AS ps
        ON get_json_object(t.meta_data, '$.policyId') = ps.base_policy_id
    WHERE t.ticket_group = 'GI_RENEWAL_CALLING'
),

filter_tagging AS (
    SELECT
        *,
        CASE
            WHEN policy_renewed_dt_helper IS NOT NULL
             AND call_connect_dt >= call_window_starts_from
            THEN 'd_1_renewed_conversations'
            WHEN policy_renewed_dt_helper IS NULL
            THEN 'd_1_non_renewed_conversations'
            ELSE NULL
        END AS filter_status
    FROM main_table
    WHERE call_connection_status = 'CONNECTED'
),

allocation_tagging AS (
    SELECT
        *,
        CASE
            WHEN filter_status = 'd_1_renewed_conversations'
             AND policy_renewed_dt_helper = date_sub(current_date(), 1)
            THEN 'dminus1'
            WHEN filter_status = 'd_1_renewed_conversations'
             AND policy_renewed_dt_helper = date_sub(current_date(), 2)
            THEN 'dminus2'
            WHEN filter_status = 'd_1_non_renewed_conversations'
             AND call_connect_dt = date_sub(current_date(), 1)
            THEN 'dminus1'
            WHEN filter_status = 'd_1_non_renewed_conversations'
             AND call_connect_dt = date_sub(current_date(), 2)
            THEN 'dminus2'
        END AS allocation_type
    FROM filter_tagging
)

SELECT DISTINCT
    crm_ticket_id,
    Agent_name,
    call_duration,
    call_connect_dt,
    NULL AS call_connect_timestamp,
    navi_customer_id,
    NULL AS policy_renewed_dt,
    L0_disposition,
    filter_status,
    allocation_type,
    policy_renewed_dt_helper
FROM allocation_tagging
WHERE filter_status = 'd_1_renewed_conversations'
  AND (
        (dayofweek(current_date()) = 2 AND allocation_type IN ('dminus1', 'dminus2'))
     OR (dayofweek(current_date()) <> 2 AND allocation_type = 'dminus1')
  )

UNION

SELECT DISTINCT
    crm_ticket_id,
    Agent_name,
    call_duration,
    call_connect_dt,
    NULL AS call_connect_timestamp,
    navi_customer_id,
    NULL AS policy_renewed_dt,
    L0_disposition,
    filter_status,
    allocation_type,
    policy_renewed_dt_helper
FROM allocation_tagging
WHERE filter_status = 'd_1_non_renewed_conversations'
  AND (
        (dayofweek(current_date()) = 2 AND allocation_type IN ('dminus1', 'dminus2'))
     OR (dayofweek(current_date()) <> 2 AND allocation_type = 'dminus1')
  )
"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1_YCxjwh-BUf8gLs-HWhD8jWIH-LPU4rq5Cz82TXWLwI/edit?gid=0#gid=0"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=3
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=3
)